# Equation Solving — From GA to Linear Systems

**Part II · Geometric Algebra** — Tutorial 11

This tutorial teaches pytanga's equation solvers: `solve`, `solve_lsq`, and
`solve_mod`. Each turns a geometric-algebra product equation (`A ∘ X = C`) into a
linear system, derives the unknown's subspace automatically, and delegates to
`numpy.linalg` (floats) or C++ Gaussian elimination (modular integers).

By the end you will be able to:

- Solve for a multiplicative inverse with `solve(A, 1)`.
- Solve general product equations `A ∘ X = C` for `GP`, `OP`, and `IP`.
- Choose `left=True` (A ∘ X) vs `left=False` (X ∘ A).
- Use `solve_lsq` for rank-deficient and over-determined systems.
- Solve modular integer systems with `solve_mod`.
- Explain the automatic blade-mask derivation pipeline.
- Select products and involutions with the `EProduct` and `EInv` enums.

> **Prerequisites:** [Tutorial 02](../02_algebra_core/) (products, inverse),
> [Tutorial 09](../09_modulus/) (`solve_mod` in context), and
> [Tutorial 10](../10_blade_mask/) (blade masks).

## 1. Setup

The three solvers are free functions in `pytanga.solver.solve`. We bind one
`BasisE3` (G(3,0), float64) for the floating-point examples.

In [1]:
from pytanga import Algebra, BladeMask, EInv, EProduct
from pytanga.basis import BasisE3
from pytanga.blade_mask.predict import inverse_blade_mask, product_blade_mask
from pytanga.solver.solve import solve, solve_lsq, solve_mod

E3 = BasisE3()

## 2. The three solvers

| Solver | Backend | When to use |
|---|---|---|
| `solve` | `numpy.linalg.solve` (square) / `lstsq` (otherwise) | float, exactly-determined |
| `solve_lsq` | `numpy.linalg.lstsq` | float, rank-deficient / over-determined |
| `solve_mod` | C++ Gaussian elimination over `Z/pZ` | integer dtypes, prime modulus |

All three share the same calling pattern and the same automatic mask pipeline:
`solve(A, C, product='gp', left=True, algebra=...)`.

## 3. Multiplicative inverse — `solve(A, 1)`

The simplest equation is `A * X = 1`. `solve(A, 1.0)` returns the multiplicative
inverse; the scalar `1.0` is coerced to a scalar multivector automatically.

In [2]:
A = E3("e1 + e2")

X = solve(A, 1.0)
X.show("X = solve(A, 1.0)")
(A * X).show("A * X")           # scalar 1

X = solve(A, 1.0): 0.5 e1 + 0.5 e2

A * X: 1

## 4. General solve — `A ∘ X = C`

For a general right-hand side, solve `A * X = C`. `product` selects the GA
operation (`'gp'`, `'op'`, `'ip'`); `left` selects `A ∘ X` (True) vs `X ∘ A`
(False).

In [3]:
C = E3("1 + 3 e1")
X = solve(A, C)
X.show("X = solve(A, 1 + 3 e1)")
(A * X).show("A * X")           # should be 1 + 3 e1

X = solve(A, 1 + 3 e1): 1.5 + 0.5 e1 + 0.5 e2 - 1.5 e12

A * X: 1 + 3 e1

In [4]:
# Outer product: e1 ^ X = e12  →  X = e2.
X_op = solve(E3("e1"), E3("e12"), product="op")
X_op.show("solve(e1, e12, product='op')")

# left=False: X ^ e1 = e12  →  X = -e2  (because e2 ^ e1 = -e12).
X_left = solve(E3("e1"), E3("e12"), product="op", left=False)
X_left.show("solve(e1, e12, product='op', left=False)")

solve(e1, e12, product='op'): e2

solve(e1, e12, product='op', left=False): - e2

## 5. Automatic blade-mask derivation

The solver never needs the unknown's subspace explicitly. Internally it derives
three masks (see [Tutorial 10](../10_blade_mask/)):

1. `a_mask = BladeMask(A)` — blades of the known operand.
2. `c_mask = BladeMask(C)` — blades of the result.
3. `b_mask = inverse_blade_mask(a_mask, c_mask)` — the maximal subspace for `X`.
4. `c_mask = product_blade_mask(a_mask, b_mask)` — grown until the system is square.

In [5]:
A = E3("e1 + e2")

a_mask = BladeMask(A)
c_mask = BladeMask(E3("1"))
b_mask = inverse_blade_mask(a_mask, c_mask)

print("a_mask :", a_mask.ids, a_mask.names())
print("c_mask :", c_mask.ids, c_mask.names())
print("b_mask :", b_mask.ids, b_mask.names())
print("square :", len(b_mask), "==", len(product_blade_mask(a_mask, b_mask)))

X = solve(A, 1.0)
X.show("X = solve(A, 1.0)")

a_mask : [1, 2] ['e1', 'e2']
c_mask : [0] ['s']
b_mask : [1, 2] ['e1', 'e2']
square : 2 == 2


X = solve(A, 1.0): 0.5 e1 + 0.5 e2

## 6. `solve_lsq` — least-squares

`solve` raises `LinAlgError` when `A` is singular. `solve_lsq` always returns the
least-squares solution. It also accepts **lists** of `A_i` / `C_i`, which it stacks
into one over-determined system for a single unknown `X`.

In [6]:
A_sing = E3("1 + e1")     # zero divisor in G(3,0): (1+e1)(1-e1) = 0

try:
    solve(A_sing, 1.0)
except Exception as e:
    print("solve raised:", type(e).__name__)

X_lsq = solve_lsq(A_sing, 1.0)
X_lsq.show("X_lsq = solve_lsq(A_sing, 1.0)")
(A_sing * X_lsq).show("A_sing * X_lsq  (least-squares, not exactly 1)")

solve raised: LinAlgError


X_lsq = solve_lsq(A_sing, 1.0): 0.25 + 0.25 e1

A_sing * X_lsq  (least-squares, not exactly 1): 0.5 + 0.5 e1

In [7]:
# Stack several equations A_i * X = C_i and solve for a single X.
A_list = [E3("e1"), E3("e2"), E3("e3")]
C_list = [E3("e1"), E3("e2"), E3("e3")]

X = solve_lsq(A_list, C_list)
X.show("X = solve_lsq([e1, e2, e3], [e1, e2, e3])")

for a, c in zip(A_list, C_list):
    print(f"  {a} * X = {a * X}   (target {c})")

X = solve_lsq([e1, e2, e3], [e1, e2, e3]): 1

  e1 * X = e1   (target e1)
  e2 * X = e2   (target e2)
  e3 * X = e3   (target e3)


## 7. `solve_mod` — modular systems

For integer-dtype algebras use `solve_mod`; it runs Gaussian elimination over
`Z/pZ`. The modulus must be prime for full invertibility. See
[Tutorial 09](../09_modulus/) for the full treatment.

In [8]:
alg_i = Algebra(3, 0, dtype="int64")
A_i = alg_i("3 e1 + 5 e2")

X_mod = solve_mod(A_i, 1, modulus=97, algebra=alg_i)
X_mod.show("X = solve_mod(A, 1, modulus=97)")
(A_i.gp_mod(X_mod, 97)).show("A * X (mod 97)")

X = solve_mod(A, 1, modulus=97): - 37 e1 + 3 e2

A * X (mod 97): 1

## 8. Enums — `EProduct` and `EInv`

`EProduct` selects the GA operation used by the solvers and matrix builders;
`EInv` selects an involution applied to an operand when building the product
matrix (`left_inv` on A, `right_inv` on X). `EInv` is a `StrEnum`, so
`EInv.REV == "rev"` is `True`.

In [9]:
print("EProduct:", [e.value for e in EProduct])
print("EInv    :", [e.value for e in EInv])
print("EInv.REV == 'rev':", EInv.REV == "rev")

EProduct: ['gp', 'ip', 'op']
EInv    : ['id', 'rev', 'conj']
EInv.REV == 'rev': True


In [10]:
from pytanga.matrix.product import product_matrix

# Build the product matrix for rev(A) * X with X conjugated.
M = product_matrix(
    E3("e1"),
    b_mask=BladeMask(E3, "e2"),
    c_mask=BladeMask(E3, "e12"),
    left_inv=EInv.REV,
    right_inv=EInv.CONJ,
)
print("product matrix shape:", M.data.shape)   # (|a|, |c|, |b|)
print("a, b, c masks      :", M.a_mask.ids, M.b_mask.ids, M.c_mask.ids)

product matrix shape: (1, 1, 1)
a, b, c masks      : [1] [2] [3]


## 9. Summary & next steps

| Task | API |
|---|---|
| Inverse | `solve(A, 1.0)` |
| General solve | `solve(A, C, product='gp', left=True)` |
| Least-squares | `solve_lsq(A, C)`, `solve_lsq([A1, A2], [C1, C2])` |
| Modular solve | `solve_mod(A, C, modulus=p, algebra=alg_i)` |
| Product selector | `EProduct.GP` / `EProduct.OP` / `EProduct.IP` |
| Involution selector | `EInv.ID` / `EInv.REV` / `EInv.CONJ` |
| Subspace derivation | `inverse_blade_mask`, `product_blade_mask` |

**Where to go next:**

- [**12 · Matrix Operations**](../12_matrix/) — the `MVMatrix` / `MVProductMatrix`
  types and `product_matrix` behind these solvers.
- [**09 · Modulus Arithmetic**](../09_modulus/) — modular arithmetic in depth.